In [ ]:
#1
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y peft
!pip install peft==0.10.0

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)


In [ ]:
!pip install transformers datasets accelerate evaluate sacrebleu evaluate

In [ ]:
#5
import os
os.listdir()

['.config', 'results', 'final-dataset.csv', 'drive', 'sample_data']

In [ ]:
#7
import pandas as pd
df = pd.read_csv("final-dataset.csv")
df.head()

,src_dogri,tgt_english
0,ओ कूडी आलसी ऐ,that girl is lazy
1,राम दा नौकर डरोकल ऐ,ram servant is timid
2,निक्की कुड़ी खेला दी ऐ,little girl is playing
3,बजार भरीदा ऐ,the market is crowded
4,तारे टिमटिमांदे न,the stars are shining


In [ ]:
#8
from datasets import load_dataset

dataset = load_dataset("csv", data_files="final-dataset.csv")
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [ ]:
#10
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer.src_lang = "doi_Deva"
tokenizer.tgt_lang = "eng_Latn"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
print(model)

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [ ]:
#11
def preprocess(example):
    inputs = tokenizer(
        example["src_dogri"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    targets = tokenizer(
        example["tgt_english"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

dataset = dataset.map(preprocess)

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

In [ ]:
#12
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 1,403,318,272 || trainable%: 0.08406132974515991


In [ ]:
#13
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-4,
    num_train_epochs=30,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    )

trainer.train()

Epoch,Training Loss,Validation Loss
1,10.421809,7.775526
2,7.777550,7.426796
3,7.571642,7.361235
4,7.491566,7.330798
5,7.447670,7.314322
6,7.417369,7.305236
7,7.395233,7.299765
8,7.380875,7.294380
9,7.368546,7.292138
10,7.359251,7.288408


KeyboardInterrupt: 

In [ ]:
#14
def translate(text):
    # This is the "no-fail" way to get the language ID
    target_lang = "eng_Latn"
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=64
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
#15
predictions = []
references = []
dogri_sentences = []

for example in dataset["test"]:
    pred = translate(example["src_dogri"])
    predictions.append(pred)
    references.append(example["tgt_english"])
    dogri_sentences.append(example["src_dogri"])

In [ ]:
import evaluate
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=references
)

print("METEOR:", meteor_score["meteor"])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: 0.5922404346908845


In [ ]:
#18

print(f"Pred: {predictions[1]}")
print(f"Ref: {references[1]}")


print(f"Type Pred: {type(predictions[1])}")
print(f"Type Ref: {type(references[1])}")

Pred: you are sad
Ref: why are you sad
Type Pred: <class 'str'>
Type Ref: <class 'str'>


In [ ]:
print("METEOR:", meteor_score["meteor"])

METEOR: 0.5922404346908845


In [ ]:
model.save_pretrained("/content/drive/MyDrive/dogri3_model")
tokenizer.save_pretrained("/content/drive/MyDrive/dogri3_model")

('/content/drive/MyDrive/dogri3_model/tokenizer_config.json',
 '/content/drive/MyDrive/dogri3_model/tokenizer.json')

In [ ]:
# 1. Type ANY Dogri sentence here (one from your CSV or a new one)
my_dogri_text = "सूहा खिन्नू मेरा ऐ"

# 2. Run the translation
output = translate(my_dogri_text)

# 3. See the result
print("--- MANUAL CHECK ---")
print(f"INPUT (Dogri): {my_dogri_text}")
print(f"OUTPUT (English): {output}")

--- MANUAL CHECK ---
INPUT (Dogri): सूहा खिन्नू मेरा ऐ
OUTPUT (English): the ball is mine


In [ ]:
import json

scores = {
    "meteor": meteor_score
}

with open("/content/drive/MyDrive/scores.json", "w") as f:
    json.dump(scores, f)